# Repeated Measures ANOVA

## Overview
Repeated measures ANOVA is used when the same subjects (or experimental units) are measured multiple times — across time points, conditions, or treatments. Because measurements on the same subject are correlated, the standard ANOVA error term is inappropriate.

**Classical RM-ANOVA** (this notebook) vs **LMM approach** (see `lmm_basics.ipynb`):

| Aspect | Classical RM-ANOVA | LMM (lme4) |
|---|---|---|
| Correlation structure | Compound symmetry | Flexible (unstructured, AR1, etc.) |
| Missing data | Requires complete cases | Handles missing data via ML |
| Sphericity | Required; correctable | Not assumed |
| Software | `aov()`, `ez`, `afex` | `lme4::lmer()` |
| Common in | Older ecological/psych literature | Modern practice |

**Quinn & Keough (2002) ch. 10–11 / Underwood (1997) ch. 12.**

---

In [ ]:
library(tidyverse); library(ez); library(afex); library(emmeans)
set.seed(21)
# Intertidal experiment: invertebrate abundance measured at
# 4 time points (Before, 1mo, 3mo, 6mo) on 15 permanent plots
n_subjects <- 15
time_pts   <- c("Before","1mo","3mo","6mo")
dat <- expand.grid(
  subject = factor(1:n_subjects),
  time    = factor(time_pts, levels=time_pts)
)
# Simulate: abundance decreases then recovers; subject random effect
subj_effect <- rep(rnorm(n_subjects, 0, 3), times=4)
time_means  <- c(20, 15, 17, 19)
dat$abundance <- time_means[as.integer(dat$time)] +
                 subj_effect + rnorm(nrow(dat), 0, 2.5)
dat$abundance <- pmax(dat$abundance, 0)
cat("Repeated measures dataset:\n"); print(head(dat, 8))
cat("\nDesign: 1 within-subjects factor (Time, 4 levels)\n")

---
## Sphericity: Mauchly's test and corrections

In [ ]:
# Classical RM-ANOVA with ez package
m_rm <- ez::ezANOVA(
  data      = dat,
  dv        = abundance,
  wid       = subject,
  within    = time,
  type      = 3,
  return_aov= TRUE
)
cat("RM-ANOVA results:\n"); print(m_rm$ANOVA)
cat("\nMauchly's Test of Sphericity:\n"); print(m_rm$`Mauchly's Test for Sphericity`)
cat("\nSphericity corrections:\n"); print(m_rm$`Sphericity Corrections`)

cat("\nInterpretation:\n")
cat("  p(Mauchly) > 0.05 → sphericity not violated → use uncorrected F\n")
cat("  p(Mauchly) < 0.05 → use GGe (Greenhouse-Geisser) or HFe (Huynh-Feldt)\n")
cat("  GGe conservative; HFe less so; GGe preferred when GGe < 0.75\n")

In [ ]:
# Post-hoc comparisons
em <- emmeans(m_rm$aov, ~ time)
cat("Estimated marginal means by time:\n"); print(em)
cat("\nPairwise comparisons (Bonferroni):\n")
print(pairs(em, adjust="bonferroni"))

# Profile plot
ggplot(dat, aes(time, abundance, group=subject)) +
  geom_line(alpha=0.3, colour="grey60") +
  stat_summary(aes(group=1), fun=mean, geom="line",
               colour="steelblue", linewidth=1.5) +
  stat_summary(fun.data=mean_se, geom="errorbar",
               colour="steelblue", width=0.2) +
  labs(title="Repeated measures: individual profiles + mean ± SE",
       x="Time", y="Abundance") + theme_minimal()

In [ ]:
# Modern alternative: LMM with lme4
library(lme4); library(lmerTest)
m_lmm <- lmer(abundance ~ time + (1|subject), data=dat)
cat("LMM equivalent (compound symmetry):\n")
print(anova(m_lmm))
cat("\nLMM vs RM-ANOVA: LMM is preferable when:\n")
cat("  - Missing data present (LMM uses all available data via ML)\n")
cat("  - Sphericity is violated (LMM can model complex covariance structures)\n")
cat("  - Unequal time intervals or continuous time variable\n")

---
## Common Pitfalls

**1. Not testing sphericity before reporting uncorrected F**
The sphericity assumption (equal variances of all pairwise difference scores) is frequently violated in ecological repeated measures data. Always report Mauchly's test and, if violated (p < 0.05), apply Greenhouse-Geisser or Huynh-Feldt correction. Modern practice often applies GGe correction by default.

**2. Using standard one-way ANOVA ignoring the within-subjects structure**
Treating repeated measures as independent observations inflates degrees of freedom and underestimates the error term, producing anti-conservative tests. The within-subjects correlation must be removed from the error term.

**3. Dropping subjects with any missing time points**
Classical RM-ANOVA in `aov()` requires a complete balanced design — subjects with missing data are excluded. This can substantially reduce power. Use `lme4::lmer()` which handles missing data via maximum likelihood and retains all available observations.

**4. Confusing within-subjects and between-subjects factors**
A within-subjects (repeated measures) factor has each subject measured at all levels. A between-subjects factor assigns each subject to only one level. Mixed designs have both. Using the wrong error term for a factor produces incorrect F-ratios.

**5. Not accounting for time as ordered when doing post-hoc tests**
For time-series repeated measures, pairwise comparisons treat all pairs equally. If a gradient over time is hypothesised, polynomial contrasts (linear, quadratic) are more powerful and appropriate.

**6. Ignoring individual subject profiles in interpretation**
The mean profile can mask important individual variation — some subjects may increase while others decrease, making the mean appear flat. Always plot individual trajectories alongside the mean.


---
*r_methods_library - Samantha McGarrigle*